# Time Anomaly Quantile Regression Proof of Concept 

## Imports

In [36]:
# Systems and Files libraries
import os
import re
from pathlib import Path

# Pandas and numpy
import pandas as pd
import numpy as np


## Merging Flights and Weather data

- The two files retrieved contain the aggregated flight and weather data respectively for the Frankfurt to Thessaloniki route (EDDF->LGTS)
- Data was compiled using the <b>parquet_aggregator.ipynb</b> notebook
- The data needs to be merged on datetime to combine flight and weather information for each flight entry's departure and arrival


### Align all flight timestamps to UTC (& round to hourly) to match weather timestamp format

In [37]:
INPUT_FILE = Path('example_data/flights.parquet')
OUTPUT_FILE = Path('example_data/flights_utc.parquet')

# Define which columns contain the Unix timestamps
FIRST_SEEN_COL = "firstSeen"
LAST_SEEN_COL = "lastSeen"
DEST_AIRPORT_COL = "estDepartureAirport"  # Column tracking departure codes


def convert_and_find_flight_timestamps():
    try:
        # Read the parquet file
        print(f"Reading {INPUT_FILE}...")
        df = pd.read_parquet(INPUT_FILE, engine="pyarrow")

        # Check if required columns exist
        required_cols = [FIRST_SEEN_COL, LAST_SEEN_COL, DEST_AIRPORT_COL]
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"Error: Columns {missing_cols} not found in the file.")
            print(f"Available columns: {list(df.columns)}")
            return

        print("Converting timestamps and generating hourly columns...")

        # Dynamically detect and handle the incoming format safely
        for col in [FIRST_SEEN_COL, LAST_SEEN_COL]:
            # Fill missing data safely before any translation steps
            df[col] = df[col].fillna(0)
            
            # If the parquet already loaded it as a datetime, drop it back to raw integer format
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                df[col] = df[col].astype("int64")
            else:
                df[col] = df[col].astype("int64")

            # Check scale: If the number is huge (19 digits long), it's nanoseconds. 
            # If it's a normal 10-digit number (starts with 16 or 17), it's seconds.
            sample_val = df[col].iloc[0] if len(df) > 0 else 0
            if sample_val > 10**11:                                         
                # Safe conversion converting from nanoseconds unit natively
                df[col] = pd.to_datetime(df[col], unit="ns", utc=True)
            else:
                # Normal conversion from standard seconds unit
                df[col] = pd.to_datetime(df[col], unit="s", utc=True)

        # Generate new rounded hourly columns cleanly using native pandas rounder - necessary to match hourly historical weather forecasts
        df["firstSeen_hourly_utc"] = df[FIRST_SEEN_COL].dt.round("h")
        df["lastSeen_hourly_utc"] = df[LAST_SEEN_COL].dt.round("h")

        # Filter for EDDF destination routes only (it is the only one in the dataset)
        print("Filtering dataset for destination 'EDDF'...")
        df = df[df[DEST_AIRPORT_COL] == "EDDF"]

        if df.empty:
            print("Warning: No flights found with estDestinationAirport 'EDDF'. Output file will be empty.")

        # Ensure output directory exists and save the file
        OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
        print(f"Saving filtered file to {OUTPUT_FILE} ({len(df)} rows)...")
        df.to_parquet(OUTPUT_FILE, engine="pyarrow", index=False)

        print("\nSuccess!")
        preview_cols = [
            DEST_AIRPORT_COL,
            FIRST_SEEN_COL,
            LAST_SEEN_COL,
            "firstSeen_hourly_utc",
            "lastSeen_hourly_utc",
        ]
        print("\nData Preview:")
        print(df[preview_cols].head())

    except FileNotFoundError:
        print(f"Error: File '{INPUT_FILE}' not found.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")


if __name__ == "__main__":
    convert_and_find_flight_timestamps()


Reading example_data\flights.parquet...
Converting timestamps and generating hourly columns...
Filtering dataset for destination 'EDDF'...
Saving filtered file to example_data\flights_utc.parquet (2081 rows)...

Success!

Data Preview:
  estDepartureAirport                 firstSeen                  lastSeen  \
0                EDDF 2023-07-31 13:00:41+00:00 2023-07-31 14:55:19+00:00   
1                EDDF 2023-07-31 10:25:08+00:00 2023-07-31 12:16:24+00:00   
2                EDDF 2023-08-01 13:05:03+00:00 2023-08-01 14:53:54+00:00   
3                EDDF 2023-08-01 10:21:02+00:00 2023-08-01 12:15:58+00:00   
4                EDDF 2023-08-02 13:16:52+00:00 2023-08-02 15:05:57+00:00   

       firstSeen_hourly_utc       lastSeen_hourly_utc  
0 2023-07-31 13:00:00+00:00 2023-07-31 15:00:00+00:00  
1 2023-07-31 10:00:00+00:00 2023-07-31 12:00:00+00:00  
2 2023-08-01 13:00:00+00:00 2023-08-01 15:00:00+00:00  
3 2023-08-01 10:00:00+00:00 2023-08-01 12:00:00+00:00  
4 2023-08-02 13:00:00

### Load data for merging

In [38]:
#load data from .parquet to pandas dataframes
weather_path = Path("example_data/weather_EDDF_LGTS.parquet")
#flights_path = Path("example_data/combined_arrivals_LGTS_utc.parquet")
flights_path = Path("example_data/flights_utc.parquet")

w_df = pd.read_parquet(weather_path, engine="pyarrow")
f_df = pd.read_parquet(flights_path, engine="pyarrow")

print(w_df.info())
print(f_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 52656 entries, 0 to 52655
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   time                        52656 non-null  datetime64[us, UTC]
 1   temperature_2m              52656 non-null  float64            
 2   relative_humidity_2m        52656 non-null  int64              
 3   dew_point_2m                52656 non-null  float64            
 4   precipitation               52656 non-null  float64            
 5   weather_code                52656 non-null  int64              
 6   pressure_msl                52656 non-null  float64            
 7   cloud_cover_low             52656 non-null  int64              
 8   cloud_cover_high            52656 non-null  int64              
 9   visibility                  52656 non-null  float64            
 10  wind_speed_10m              52656 non-null  float64            
 11  

In [39]:
# check for duplicates in the weather dataframe based on airport and time columns
print(w_df.duplicated(["airport", "time"]).sum())

0


### Merge 

In [40]:

# Work on copies

flights = f_df.copy()
weather = w_df.copy()

# A timestamp bug occurred in the flights dataset
# It was identified when data type casting to different timestamp formats (ns vs us) resulted in dates set in 1970 when they were expected to be in 2023
# The 1970 bug is caused by unix timestamps being referenced to the epoch (1970-01-01 00:00:00 UTC) and can occur when timestamps are not properly handled or converted
# To avoid this: ensure that all datetime columns are parsed as timezone-aware UTC datetimes.


# We do this for both dataframes to ensure everything lines up correctly and avoids any potential issues with timezone differences or misalignment of timestamps

for col in ["firstSeen", "lastSeen", "firstSeen_hourly_utc", "lastSeen_hourly_utc"]:
    if col in flights.columns:
        flights[col] = pd.to_datetime(flights[col], utc=True)


if "time" in weather.columns:
    weather["time"] = pd.to_datetime(weather["time"], utc=True)


# Converting to strings completely strips out internal sub-second precision and unit differences (ns vs us) 
# between your data files is now eliminated, and the merge keys will match correctly

flights["dep_merge_key"] = flights["firstSeen_hourly_utc"].dt.strftime("%Y-%m-%d %H:00")
flights["arr_merge_key"] = flights["lastSeen_hourly_utc"].dt.strftime("%Y-%m-%d %H:00")
weather["weather_merge_key"] = weather["time"].dt.strftime("%Y-%m-%d %H:00")

# The weather features we want to merge into the flights dataframe are defined in a list for easy reference and manipulation
weather_features = [
    "temperature_2m", 
    "relative_humidity_2m", 
    "dew_point_2m", 
    "precipitation",
    "weather_code", 
    "pressure_msl", 
    "cloud_cover_low", 
    "cloud_cover_high",
    "visibility", 
    "wind_speed_10m", 
    "wind_speed_180m", 
    "wind_direction_10m",
    "wind_direction_180m", 
    "cape", 
    "geopotential_height_850hPa",
]

# Verify that there are no duplicates in the weather dataframe based on the airport and weather_merge_key columns
weather_duplicates = weather.duplicated(subset=["airport", "weather_merge_key"]).sum()

print("=" * 70)
print("WEATHER KEY VALIDATION")
print("=" * 70)
print(f"Duplicate airport + time combinations: {weather_duplicates}")

if weather_duplicates > 0:
    print("Dropping duplicates found in historical weather source arrays...")
    weather = weather.drop_duplicates(subset=["airport", "weather_merge_key"], keep="first")




# MERGE WEATHER DATA INTO FLIGHTS DATAFRAME ------------------------------------------------------------------------------------------------

# For this size of dataset, we could do two sequential merges: one for departure weather and one for arrival weather
# However, larger datasets may require a more memory-efficient approach, such as dual-indexed joins

# Main idea: 
# Our left table is flights 
# And we lookup through weather using a MultiIndex of (airport, weather_merge_key) to find the corresponding weather features for both departure and arrival airports
# The index allows really quick lookups

# Why merge and not join?
# Merge allows us to match regular columns (left_on) directly to the weather index (right_index=True)
# This avoids having to constantly set and reset indices on the flights dataframe, keeping the original column layout intact


# Prepare base weather index -> create a multiindex of airport and weather_merge_key, and select only the weather features we want to merge into the flights dataframe
w_indexed = weather.set_index(["airport", "weather_merge_key"])[
    weather_features
]

# Left join departure weather on ("airport", "weather_merge_key") == ("estDepartureAirport", "dep_merge_key")
# Right_index=True tells pandas to match the flights keys directly against the fast weather MultiIndex
# This retains the speed of index lookups while keeping the original flight column order intact
flights = flights.merge(
    w_indexed.add_prefix("dep_"),
    left_on=["estDepartureAirport", "dep_merge_key"],
    right_index=True,
    how="left",
)

# Left join arrival weather on ("airport", "weather_merge_key") == ("estArrivalAirport", "arr_merge_key")
# Right_index=True leverages the same pre-built weather lookup index for maximum memory efficiency
flights = flights.merge(
    w_indexed.add_prefix("arr_"),
    left_on=["estArrivalAirport", "arr_merge_key"],
    right_index=True,
    how="left",
)

# ---------------------------------------------------------------------------------------------------------------------------

# Validate the merge by checking for missing weather data in the flights dataframe
print()
print("=" * 70)
print("MERGE VALIDATION")
print("=" * 70)
print(f"Final number of flights: {len(flights)}")

missing_departure_weather = flights["dep_temperature_2m"].isna().sum()
missing_arrival_weather = flights["arr_temperature_2m"].isna().sum()

print(f"Flights missing departure weather: {missing_departure_weather} / {len(flights)}")
print(f"Flights missing arrival weather:   {missing_arrival_weather} / {len(flights)}")

print()
print("=" * 70)
print("FINAL DATASET INFO SUMMARY")
print("=" * 70)
flights.info()


WEATHER KEY VALIDATION
Duplicate airport + time combinations: 0

MERGE VALIDATION
Final number of flights: 2081
Flights missing departure weather: 0 / 2081
Flights missing arrival weather:   0 / 2081

FINAL DATASET INFO SUMMARY
<class 'pandas.DataFrame'>
RangeIndex: 2081 entries, 0 to 2080
Data columns (total 48 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   icao24                            2081 non-null   str                
 1   firstSeen                         2081 non-null   datetime64[ms, UTC]
 2   estDepartureAirport               2081 non-null   str                
 3   lastSeen                          2081 non-null   datetime64[ms, UTC]
 4   estArrivalAirport                 2081 non-null   str                
 5   callsign                          2081 non-null   str                
 6   estDepartureAirportHorizDistance  2081 non-null   float64            
 

## Training Data Preparation


- Calculate time duration for each flight (will be necessary to derive the target)
- The target is time_anomaly = flight_duration - median_flight_duration 
    - It is NOT calculated here because it must be calculated for each time split fold to respect the temporal order of data and avoid data leakage
- Remove any extreme outliers that are suspected to not reflect actual flight anomalies
- Establish "route" column, this is an extendability feature for the script, if it is used for datasets where this can help with grouping according to route
- Drop columns that are:
    - low variance/constant
    - categorical without useful information 
    - going to contribute to data leakage.
        - here these are features related to flight duration because the target is derived from that

In [41]:

# Ensure absolute chronological sorting
flights = flights.sort_values("firstSeen").reset_index(drop=True)

# Extract base flight duration and drop missing values
flights["flight_duration"] = (
    flights["lastSeen"] - flights["firstSeen"]
).dt.total_seconds() / 60.0
flights = flights.dropna(subset=["flight_duration"])

# --- OUTLIER REMOVAL RULE ---
# Create a mask for the specific Spring 2025 extreme outlier (> 300 minutes)
# This removes the data logging error before it splits into Fold 1's test set.  
# it is highly unlikely that the flight duration would exceed 5 hours, transpoder data logging error is likely the cause of this extreme outlier.
outlier_mask = (
    (flights["firstSeen"] >= "2025-04-01")
    & (flights["firstSeen"] <= "2025-07-01")
    & (flights["flight_duration"] > 300)
)

print(f"Removing {outlier_mask.sum()} extreme outlier row(s) from Spring 2025...")
flights = flights[~outlier_mask].reset_index(drop=True)
# ----------------------------

# Create route identifier 
if "route" not in flights.columns:
    flights["route"] = (
        flights["estDepartureAirport"].astype(str)
        + "_"
        + flights["estArrivalAirport"].astype(str)
    )

# Feature Engineering
flights["airline"] = flights["callsign"].str[:3]
flights["hour_of_day"] = flights["firstSeen"].dt.hour
flights["month"] = flights["firstSeen"].dt.month

# Keep a clean reference of flight dates for TimeSeriesSplit visualization/logging
flights["flight_date_parsed"] = pd.to_datetime(flights["firstSeen"].dt.date)

# Define Columns to Drop (Keep raw variables needed for loop-level target calculation)
cols_to_drop = [
    "icao24",
    "firstSeen",
    "lastSeen",
    "estDepartureAirport",
    "estArrivalAirport",
    "callsign",
    "flight_date",
    "firstSeen_hourly_utc",
    "lastSeen_hourly_utc",
    "dep_merge_key",
    "arr_merge_key",
    "departureAirportCandidatesCount",
    "arrivalAirportCandidatesCount",
    "estDepartureAirportHorizDistance",
    "estDepartureAirportVertDistance",
    "estArrivalAirportHorizDistance",
    "estArrivalAirportVertDistance",
]

# Drop useless columns
X_raw = flights.drop(columns=[c for c in cols_to_drop if c in flights.columns])

# Separate Categorical and Numerical tracking lists
categorical_cols = ["airline"]
numeric_cols = [
    col
    for col in X_raw.columns
    if col not in categorical_cols
    and col not in ["flight_duration", "route", "flight_date_parsed"]
]

# Drop low variance/constant features globally based on independent columns
constant_cols = [
    c
    for c in numeric_cols
    if X_raw[c].nunique() <= 1 and c not in ["hour_of_day", "month"]
]
if constant_cols:
    X_raw = X_raw.drop(columns=constant_cols)
    numeric_cols = [c for c in numeric_cols if c not in constant_cols]


Removing 1 extreme outlier row(s) from Spring 2025...


In [42]:
flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 2080 entries, 0 to 2079
Data columns (total 53 columns):
 #   Column                            Non-Null Count  Dtype              
---  ------                            --------------  -----              
 0   icao24                            2080 non-null   str                
 1   firstSeen                         2080 non-null   datetime64[ms, UTC]
 2   estDepartureAirport               2080 non-null   str                
 3   lastSeen                          2080 non-null   datetime64[ms, UTC]
 4   estArrivalAirport                 2080 non-null   str                
 5   callsign                          2080 non-null   str                
 6   estDepartureAirportHorizDistance  2080 non-null   float64            
 7   estDepartureAirportVertDistance   2080 non-null   float64            
 8   estArrivalAirportHorizDistance    2080 non-null   int64              
 9   estArrivalAirportVertDistance     2080 non-null   int64              
 10 

## Time-Series K-fold cross validation of LightGBM Quantile Regressor

In [43]:
import lightgbm as lgb
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    root_mean_squared_error,
)
from sklearn.model_selection import TimeSeriesSplit

# Configure TimeSeriesSplit on our chronologically sorted data
tscv = TimeSeriesSplit(n_splits=3)
target_quantile = 0.50  # Change to 0.90 or 0.95 when testing upper quantiles

print(
    f"Production Validation: Training LightGBM on 3-Year Chronological Data ({X_raw.shape[0]} rows)..."
)

fold = 1
for train_idx, test_idx in tscv.split(X_raw):

    # Slice raw chunks chronologically
    train_chunk = X_raw.iloc[train_idx].copy()
    test_chunk = X_raw.iloc[test_idx].copy()

    # Track operational dates for logging
    train_dates = train_chunk["flight_date_parsed"]
    days_trained = (train_dates.max() - train_dates.min()).days

    print(f"\n--- FOLD {fold} CONFIGURATION ---")
    print(
        f"Training Window: {train_dates.min().strftime('%Y-%m-%d')} to {train_dates.max().strftime('%Y-%m-%d')} ({days_trained} days)"
    )

    # LEAKAGE PREVENTION: Compute route medians using ONLY training fold history and then drop flight duration and median_flight_time
    # Apply route medians directly to train/test frames
    train_chunk["median_flight_time"] = train_chunk["route"].map(route_medians)
    test_chunk["median_flight_time"] = test_chunk["route"].map(route_medians)

    # Calculate target variable (Time Anomaly) dynamically inside the fold
    y_train = train_chunk["flight_duration"] - train_chunk["median_flight_time"]
    y_test = test_chunk["flight_duration"] - test_chunk["median_flight_time"]
    
    #  Calculate target variable (Time Anomaly) dynamically inside the fold
    y_train = train_chunk["flight_duration"] - train_chunk["median_flight_time"]
    y_test = test_chunk["flight_duration"] - test_chunk["median_flight_time"]

    #  Final Feature Matrix Assembly & Isolation
    X_train_raw = train_chunk.drop(
        columns=["flight_duration", "route", "flight_date_parsed", "median_flight_time"]
    ).copy()
    X_test_raw = test_chunk.drop(
        columns=["flight_duration", "route", "flight_date_parsed","median_flight_time"]
    ).copy()


    # Robust Target Encoding mapped strictly to the target objective quantile
    X_train_enc = X_train_raw.copy()
    X_test_enc = X_test_raw.copy()
    global_fold_quantile = y_train.quantile(target_quantile)

    for col in categorical_cols:
        quantile_map = y_train.groupby(X_train_raw[col]).quantile(target_quantile)
        X_train_enc[col] = (
            X_train_raw[col].map(quantile_map).fillna(global_fold_quantile)
        )
        X_test_enc[col] = (
            X_test_raw[col].map(quantile_map).fillna(global_fold_quantile)
        )

    # Safe Missing Value Imputation using Train Set statistics
    imputer = SimpleImputer(strategy="median")
    X_train_clean = pd.DataFrame(
        imputer.fit_transform(X_train_enc), columns=X_train_enc.columns
    )
    X_test_clean = pd.DataFrame(
        imputer.transform(X_test_enc), columns=X_test_enc.columns
    )

    # Train LightGBM Quantile Regressor
    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=target_quantile,
        n_estimators=200,
        learning_rate=0.015,
        max_depth=2,
        num_leaves=3,
        subsample=0.5,
        subsample_freq=1,
        colsample_bytree=0.4,
        reg_alpha=10.0,
        reg_lambda=10.0,
        random_state=42,
        verbose=-1,
    )

    model.fit(X_train_clean, y_train)
    y_pred = model.predict(X_test_clean)
    

    # Evaluate Metrics
    # Standard regression residual: Prediction - Actual
    # Positive = Over-predicting | Negative = Under-predicting
    residuals = y_pred - y_test  
    
    # Calculate errors using standard scikit-learn functions
    mae = mean_absolute_error(y_test, y_pred)
    med_ae = median_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    # Pinball loss 
    pinball_loss = np.mean(
        np.maximum(target_quantile * (-residuals), (target_quantile - 1) * (-residuals))
    )
    
    under_pred_pct = np.mean(y_test <= y_pred) * 100
    mean_bias = np.mean(residuals) # Positive means model over-predicts

    print(f"Pinball Loss:                        {pinball_loss:.4f}")
    print(f"Mean Bias Error (Pred - True):       {mean_bias:.2f} minutes")
    print(f"Actuals Under Prediction:            {under_pred_pct:.1f}% (Ideal: {target_quantile * 100}%)")
    print(f"[Diagnostic] Mean Absolute Error:    {mae:.2f} minutes")
    print(f"[Diagnostic] Median Absolute Error:  {med_ae:.2f} minutes")
    print(f"[Diagnostic] Root Mean Squared Error:{rmse:.2f} minutes")
    print("-" * 55)

    fold += 1




Production Validation: Training LightGBM on 3-Year Chronological Data (2080 rows)...

--- FOLD 1 CONFIGURATION ---
Training Window: 2023-07-31 to 2024-07-12 (347 days)
Pinball Loss:                        2.5892
Mean Bias Error (Pred - True):       -1.60 minutes
Actuals Under Prediction:            41.0% (Ideal: 50.0%)
[Diagnostic] Mean Absolute Error:    5.18 minutes
[Diagnostic] Median Absolute Error:  4.34 minutes
[Diagnostic] Root Mean Squared Error:6.51 minutes
-------------------------------------------------------

--- FOLD 2 CONFIGURATION ---
Training Window: 2023-07-31 to 2025-04-30 (639 days)
Pinball Loss:                        2.4192
Mean Bias Error (Pred - True):       -0.36 minutes
Actuals Under Prediction:            51.0% (Ideal: 50.0%)
[Diagnostic] Mean Absolute Error:    4.84 minutes
[Diagnostic] Median Absolute Error:  3.91 minutes
[Diagnostic] Root Mean Squared Error:6.39 minutes
-------------------------------------------------------

--- FOLD 3 CONFIGURATION ---
T

## Time-Series K-fold cross validation of XGBoost Quantile Regressor

In [44]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    median_absolute_error,
    root_mean_squared_error,
)
from sklearn.model_selection import TimeSeriesSplit

# Configure TimeSeriesSplit on our chronologically sorted data
tscv = TimeSeriesSplit(n_splits=3)
target_quantile = 0.50  # Change to 0.90 or 0.95 when testing upper quantiles

print(
    f"Production Validation: Training XGBoost on 3-Year Chronological Data ({X_raw.shape[0]} rows)..."
)

fold = 1
for train_idx, test_idx in tscv.split(X_raw):

    # Slice raw chunks chronologically
    train_chunk = X_raw.iloc[train_idx].copy()
    test_chunk = X_raw.iloc[test_idx].copy()

    # Track operational dates for logging - this is important for understanding the temporal coverage of each fold
    train_dates = train_chunk["flight_date_parsed"]
    days_trained = (train_dates.max() - train_dates.min()).days

    print(f"\n--- FOLD {fold} CONFIGURATION ---")
    print(
        f"Training Window: {train_dates.min().strftime('%Y-%m-%d')} to {train_dates.max().strftime('%Y-%m-%d')} ({days_trained} days)"
    )

    # LEAKAGE PREVENTION: Compute route medians using ONLY training fold history and then drop flight duration and median_flight_time
    route_medians = train_chunk.groupby("route")["flight_duration"].median()

    # Apply route medians directly to train/test frames
    train_chunk["median_flight_time"] = train_chunk["route"].map(route_medians)
    test_chunk["median_flight_time"] = test_chunk["route"].map(route_medians)

    # Calculate target variable (Time Anomaly) dynamically inside the fold
    y_train = train_chunk["flight_duration"] - train_chunk["median_flight_time"]
    y_test = test_chunk["flight_duration"] - test_chunk["median_flight_time"]

    # Final Feature Matrix Assembly & Isolation
    X_train_raw = train_chunk.drop(
        columns=["flight_duration", "route", "flight_date_parsed","median_flight_time"]
    ).copy()
    X_test_raw = test_chunk.drop(
        columns=["flight_duration", "route", "flight_date_parsed","median_flight_time"]
    ).copy()

    # Robust Target Encoding mapped strictly to the target objective quantile
    X_train_enc = X_train_raw.copy()
    X_test_enc = X_test_raw.copy()
    global_fold_quantile = y_train.quantile(target_quantile)

    for col in categorical_cols:
        quantile_map = y_train.groupby(X_train_raw[col]).quantile(target_quantile)
        X_train_enc[col] = (
            X_train_raw[col].map(quantile_map).fillna(global_fold_quantile)
        )
        X_test_enc[col] = (
            X_test_raw[col].map(quantile_map).fillna(global_fold_quantile)
        )

    # Safe Missing Value Imputation using Train Set statistics
    imputer = SimpleImputer(strategy="median")
    X_train_clean = pd.DataFrame(
        imputer.fit_transform(X_train_enc), columns=X_train_enc.columns
    )
    X_test_clean = pd.DataFrame(
        imputer.transform(X_test_enc), columns=X_test_enc.columns
    )

    # Train XGBoost Quantile Regressor
    model = xgb.XGBRegressor(
        objective="reg:quantileerror",
        quantile_alpha=target_quantile,
        n_estimators=200,
        learning_rate=0.015,
        max_depth=2,               # num_leaves doesn't exist in XGBoost depth-wise trees by default
        subsample=0.5,
        colsample_bytree=0.4,
        alpha=10.0,                # Equivalent to LightGBM reg_alpha (L1)
        reg_lambda=10.0,           # Equivalent to LightGBM reg_lambda (L2)
        random_state=42,
    )

    model.fit(X_train_clean, y_train)
    y_pred = model.predict(X_test_clean)

    # Evaluate Metrics
    # Standard regression residual: Prediction - Actual
    # Positive = Over-predicting | Negative = Under-predicting
    residuals = y_pred - y_test  
    
    # Calculate errors using standard scikit-learn functions
    mae = mean_absolute_error(y_test, y_pred)
    med_ae = median_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    
    # Pinball loss 
    pinball_loss = np.mean(
        np.maximum(target_quantile * (-residuals), (target_quantile - 1) * (-residuals))
    )
    
    under_pred_pct = np.mean(y_test <= y_pred) * 100
    mean_bias = np.mean(residuals) # Positive means model over-predicts

    print(f"Pinball Loss:                        {pinball_loss:.4f}")
    print(f"Mean Bias Error (Pred - True):       {mean_bias:.2f} minutes")
    print(f"Actuals Under Prediction:            {under_pred_pct:.1f}% (Ideal: {target_quantile * 100}%)")
    print(f"[Diagnostic] Mean Absolute Error:    {mae:.2f} minutes")
    print(f"[Diagnostic] Median Absolute Error:  {med_ae:.2f} minutes")
    print(f"[Diagnostic] Root Mean Squared Error:{rmse:.2f} minutes")
    print("-" * 55)

    fold += 1



Production Validation: Training XGBoost on 3-Year Chronological Data (2080 rows)...

--- FOLD 1 CONFIGURATION ---
Training Window: 2023-07-31 to 2024-07-12 (347 days)
Pinball Loss:                        2.5368
Mean Bias Error (Pred - True):       -1.48 minutes
Actuals Under Prediction:            41.3% (Ideal: 50.0%)
[Diagnostic] Mean Absolute Error:    5.07 minutes
[Diagnostic] Median Absolute Error:  4.22 minutes
[Diagnostic] Root Mean Squared Error:6.39 minutes
-------------------------------------------------------

--- FOLD 2 CONFIGURATION ---
Training Window: 2023-07-31 to 2025-04-30 (639 days)
Pinball Loss:                        2.4522
Mean Bias Error (Pred - True):       -0.45 minutes
Actuals Under Prediction:            49.8% (Ideal: 50.0%)
[Diagnostic] Mean Absolute Error:    4.90 minutes
[Diagnostic] Median Absolute Error:  4.00 minutes
[Diagnostic] Root Mean Squared Error:6.46 minutes
-------------------------------------------------------

--- FOLD 3 CONFIGURATION ---
Tr

In [45]:
X_train_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 1560 entries, 0 to 1559
Data columns (total 33 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   dep_temperature_2m              1560 non-null   float64
 1   dep_relative_humidity_2m        1560 non-null   float64
 2   dep_dew_point_2m                1560 non-null   float64
 3   dep_precipitation               1560 non-null   float64
 4   dep_weather_code                1560 non-null   float64
 5   dep_pressure_msl                1560 non-null   float64
 6   dep_cloud_cover_low             1560 non-null   float64
 7   dep_cloud_cover_high            1560 non-null   float64
 8   dep_visibility                  1560 non-null   float64
 9   dep_wind_speed_10m              1560 non-null   float64
 10  dep_wind_speed_180m             1560 non-null   float64
 11  dep_wind_direction_10m          1560 non-null   float64
 12  dep_wind_direction_180m         1560 non-null